# Lab5: 多轮迭代式框+文本提示分割 + 点精修 & Instance ID
## Iterative Box-Text Segmentation with Point Refinement and Instance Tracking

---

### 新增特性

1. **多轮分割**：同一张图反复添加框+文本提示，结果累积
2. **Instance ID**：每个构件分配唯一实例编号（如 `swallowtail#1`）
3. **点精修**：对已有掩码添加「正面点（扩大）」和「负面点（缩小）」精细化调整
4. **鼠标点击交互**：交互式 matplotlib 画布，左键=正面点，右键=负面点
5. **精修后替换**：新掩码替换旧的，Instance ID 保持不变
6. **批量导出**：导出全部可见掩码

In [3]:
# ========== 环境准备 ==========
# 安装交互式 matplotlib 支持（用于鼠标点击精修）
import subprocess, sys, importlib
try:
    import ipympl
    print("[✅] ipympl 已安装")
except ImportError:
    print("[⏳] 正在安装 ipympl (交互式 matplotlib)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipympl"])
    print("[✅] ipympl 安装完成，请重启 Kernel")

# 启用交互式 matplotlib
%matplotlib widget

import os, sys, csv, json, warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path

from IPython.display import display
import ipywidgets as widgets

# 中文字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'SimSun',
                                     'FangSong', 'Arial Unicode MS', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

print("[✅] 环境就绪")

[✅] ipympl 已安装
[✅] 环境就绪


In [4]:
# ========== 路径配置 ==========
_cwd = Path.cwd().resolve()
if (_cwd / "sam3").is_dir():
    REPO_ROOT = _cwd
elif (_cwd.parent / "sam3").is_dir():
    REPO_ROOT = _cwd.parent
else:
    raise RuntimeError(f"Cannot find repo root from {_cwd}")

CKPT_PATH = REPO_ROOT / "sam3.pt"
INPUT_DIR = REPO_ROOT / "Inputs" / "RawImages"
CSV_PATH = INPUT_DIR / "试标注图像清单.csv"
LAB5_OUTPUT = REPO_ROOT / "Outputs" / "Lab5_output"
LAB5_OUTPUT.mkdir(parents=True, exist_ok=True)

if not CKPT_PATH.exists():
    raise FileNotFoundError(f"权重未找到: {CKPT_PATH}")

image_notes = {}
if CSV_PATH.exists():
    with open(CSV_PATH, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            image_notes[row["编号"]] = row

image_files = sorted([f for f in os.listdir(INPUT_DIR) if f.endswith("_RawImage.png")])
print(f"图片数量: {len(image_files)}")
print(f"输出目录: {LAB5_OUTPUT}")
print(f"权重文件: {CKPT_PATH}")

图片数量: 16
输出目录: C:\Users\Legion\Desktop\WIE\SAM3_Labs\Outputs\Lab5_output
权重文件: C:\Users\Legion\Desktop\WIE\SAM3_Labs\sam3.pt


In [5]:
# ========== 加载 SAM 3 模型 ==========
print("[⏳] 正在加载 SAM 3 模型...")
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"    设备: {device}")

model = build_sam3_image_model(
    checkpoint_path=str(CKPT_PATH),
    load_from_HF=False,
    device=device,
    eval_mode=True,
)
processor = Sam3Processor(model, device=device)
print(f"[✅] SAM 3 加载完成 | 参数: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

[⏳] 正在加载 SAM 3 模型...
    设备: cuda
[✅] SAM 3 加载完成 | 参数: 841M


In [6]:
# ========== Lab5: 多轮分割 + Instance ID + 点精修 ==========

# ---- 全局状态 ----
_current_img = None
_current_img_id = None
_current_img_w = 800
_current_img_h = 600
seg_entries = []       # {id, instance_id, text, box, mask, score, visible, pts_history}
next_entry_id = 0
_instance_counters = {}  # {"swallowtail ridge": 3}
_refining_entry_id = None
_refine_pts_pos = []    # [(x, y), ...]
_refine_pts_neg = []    # [(x, y), ...]
_refine_fig = None
_refine_ax = None
_refine_scatter_pos = None
_refine_scatter_neg = None
_cmap = plt.cm.tab10

plt.ioff()  # 默认关闭交互，需要时才打开


def _get_color(eid):
    return _cmap(eid % 10)


def _get_instance_id(text):
    global _instance_counters
    _instance_counters[text] = _instance_counters.get(text, 0) + 1
    return f"{text}#{_instance_counters[text]}"


def load_image(img_id):
    global _current_img, _current_img_id, _current_img_w, _current_img_h
    global seg_entries, next_entry_id, _instance_counters

    filename = f"{img_id}_RawImage.png"
    img_path = INPUT_DIR / filename
    if not img_path.exists():
        return

    _current_img = Image.open(img_path).convert("RGB")
    _current_img_id = img_id
    _current_img_w, _current_img_h = _current_img.size

    seg_entries = []
    next_entry_id = 0
    _instance_counters = {}
    cancel_refinement()

    slider_x2.max = _current_img_w
    slider_y2.max = _current_img_h
    slider_x1.max = max(0, _current_img_w - 10)
    slider_y1.max = max(0, _current_img_h - 10)

    notes = image_notes.get(img_id, {})
    desc_output.clear_output()
    with desc_output:
        print(f"[📷] {filename} ({_current_img_w}x{_current_img_h})")
        for k in ["主要构件", "场景特点", "干扰项"]:
            val = notes.get(k, "")
            if val:
                print(f"  - {val}")

    update_preview(None)
    rebuild_mask_list()
    redraw_visualization()


def update_preview(change):
    if _current_img is None:
        return
    x1, y1 = slider_x1.value, slider_y1.value
    x2, y2 = slider_x2.value, slider_y2.value
    preview_output.clear_output(wait=True)
    with preview_output:
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(np.array(_current_img))
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1,
                             linewidth=2, edgecolor='lime', facecolor='lime', alpha=0.2)
        ax.add_patch(rect)
        cx = ((x1+x2)/2)/_current_img_w
        cy = ((y1+y2)/2)/_current_img_h
        bw = (x2-x1)/_current_img_w
        bh = (y2-y1)/_current_img_h
        ax.set_title(f"Norm: cx={cx:.3f}, cy={cy:.3f}, w={bw:.3f}, h={bh:.3f}")
        ax.axis("off")
        plt.tight_layout()
        plt.show()
        plt.close(fig)


def run_segmentation(btn):
    global seg_entries, next_entry_id, _instance_counters
    if _current_img is None:
        return
    text_prompt = text_input.value.strip()
    if not text_prompt:
        print("请输入文本提示")
        return
    x1, y1 = slider_x1.value, slider_y1.value
    x2, y2 = slider_x2.value, slider_y2.value
    cx = ((x1+x2)/2)/_current_img_w
    cy = ((y1+y2)/2)/_current_img_h
    bw = (x2-x1)/_current_img_w
    bh = (y2-y1)/_current_img_h

    result_output.clear_output(wait=True)
    with result_output:
        print(f"[⏳] Running... text='{text_prompt}'")
    try:
        state = processor.set_image(_current_img)
        state = processor.set_text_prompt(prompt=text_prompt, state=state)
        state = processor.add_geometric_prompt(
            box=[cx, cy, bw, bh], label=True, state=state)
        masks, scores = state["masks"], state["scores"]

        new_count = 0
        for i in range(len(masks)):
            instance_id = _get_instance_id(text_prompt)
            seg_entries.append({
                "id": next_entry_id,
                "instance_id": instance_id,
                "text": text_prompt,
                "box": (x1, y1, x2, y2),
                "mask": masks[i].squeeze().cpu(),
                "score": scores[i].item(),
                "visible": True,
                "pts_history": {"pos": [], "neg": []},
            })
            next_entry_id += 1
            new_count += 1

        result_output.clear_output(wait=True)
        with result_output:
            print(f"[✅] 新增 {new_count} 个掩码 | 总计 {len(seg_entries)}")
        rebuild_mask_list()
        redraw_visualization()
    except Exception as e:
        import traceback; traceback.print_exc()
        result_output.clear_output(wait=True)
        with result_output:
            print(f"[❌] 推理失败: {e}")


# ---- 精修核心逻辑 ----

def start_refinement(entry_id):
    global _refining_entry_id, _refine_pts_pos, _refine_pts_neg
    global _refine_fig, _refine_ax

    entry = _find_entry(entry_id)
    if entry is None:
        return

    _refining_entry_id = entry_id
    _refine_pts_pos = list(entry["pts_history"]["pos"])
    _refine_pts_neg = list(entry["pts_history"]["neg"])

    refine_title.value = f"<b>精修: #{entry['id']} {entry['instance_id']}</b> (score={entry['score']:.3f})"
    refine_info.value = f"⭐ 正面: {len(_refine_pts_pos)} 点 | ❌ 负面: {len(_refine_pts_neg)} 点"
    _draw_refine_figure()
    refine_panel.layout.visibility = "visible"


def _find_entry(entry_id):
    for e in seg_entries:
        if e["id"] == entry_id:
            return e
    return None


def _draw_refine_figure():
    global _refine_fig, _refine_ax, _refine_scatter_pos, _refine_scatter_neg
    entry = _find_entry(_refining_entry_id)
    if entry is None or _current_img is None:
        return

    refine_fig_output.clear_output(wait=True)
    with refine_fig_output:
        plt.ioff()
        _refine_fig, _refine_ax = plt.subplots(figsize=(10, 8))
        _refine_ax.imshow(np.array(_current_img))

        mask_np = entry["mask"].numpy()
        overlay = np.zeros((*mask_np.shape, 4))
        c = _get_color(entry["id"])
        overlay[mask_np] = [*c[:3], 0.4]
        _refine_ax.imshow(overlay)

        # 已有精修点
        if _refine_pts_pos:
            xs, ys = zip(*_refine_pts_pos)
            _refine_scatter_pos = _refine_ax.scatter(xs, ys, s=200,
                marker='o', facecolors='lime', edgecolors='darkgreen', linewidths=2, zorder=5)
        if _refine_pts_neg:
            xs, ys = zip(*_refine_pts_neg)
            _refine_scatter_neg = _refine_ax.scatter(xs, ys, s=250,
                marker='X', facecolors='red', edgecolors='darkred', linewidths=2, zorder=5)

        _refine_ax.set_title(f"精修: #{entry['id']} {entry['instance_id']} | 左键=正面 右键=负面")
        _refine_ax.axis("off")

        def _on_click(event):
            if event.inaxes != _refine_ax:
                return
            x, y = int(event.xdata), int(event.ydata)
            if event.button == 1:  # 左键 = 正面
                _refine_pts_pos.append((x, y))
            elif event.button == 3:  # 右键 = 负面
                _refine_pts_neg.append((x, y))
            else:
                return
            _update_refine_scatter()
            refine_info.value = f"⭐ 正面: {len(_refine_pts_pos)} 点 | ❌ 负面: {len(_refine_pts_neg)} 点"

        _refine_fig.canvas.mpl_connect('button_press_event', _on_click)
        plt.ion()
        plt.show()


def _update_refine_scatter():
    global _refine_scatter_pos, _refine_scatter_neg
    if _refine_fig is None:
        return
    if _refine_scatter_pos:
        _refine_scatter_pos.remove()
    if _refine_scatter_neg:
        _refine_scatter_neg.remove()

    if _refine_pts_pos:
        xs, ys = zip(*_refine_pts_pos)
        _refine_scatter_pos = _refine_ax.scatter(xs, ys, s=200,
            marker='o', facecolors='lime', edgecolors='darkgreen', linewidths=2, zorder=5)
    if _refine_pts_neg:
        xs, ys = zip(*_refine_pts_neg)
        _refine_scatter_neg = _refine_ax.scatter(xs, ys, s=250,
            marker='X', facecolors='red', edgecolors='darkred', linewidths=2, zorder=5)
    _refine_fig.canvas.draw_idle()


def apply_refinement(btn):
    if _refining_entry_id is None:
        return
    entry = _find_entry(_refining_entry_id)
    if entry is None or _current_img is None:
        return

    # 没有新点则跳过
    all_pos = _refine_pts_pos + list(entry["pts_history"]["pos"])
    all_neg = _refine_pts_neg + list(entry["pts_history"]["neg"])
    # 去重
    pos_set = list(dict.fromkeys(all_pos))
    neg_set = list(dict.fromkeys(all_neg))
    # 去除既在正又在负的点
    overlap = set(pos_set) & set(neg_set)
    pos_set = [p for p in pos_set if p not in overlap]
    neg_set = [n for n in neg_set if n not in overlap]

    if not pos_set and not neg_set:
        with refine_result_output:
            print("[⚠] 请至少添加一个精修点")
        return

    x1, y1, x2, y2 = entry["box"]
    cx = ((x1+x2)/2)/_current_img_w
    cy = ((y1+y2)/2)/_current_img_h
    bw = (x2-x1)/_current_img_w
    bh = (y2-y1)/_current_img_h

    with refine_result_output:
        print(f"[⏳] 精修中... 正面={len(pos_set)}, 负面={len(neg_set)}")

    try:
        state = processor.set_image(_current_img)
        state = processor.set_text_prompt(prompt=entry["text"], state=state)
        state = processor.add_geometric_prompt(
            box=[cx, cy, bw, bh], label=True, state=state)

        # 添加精修点
        if pos_set:
            pts_norm = [[p[0]/_current_img_w, p[1]/_current_img_h] for p in pos_set]
            pts_t = torch.tensor(pts_norm, device=device, dtype=torch.float32).view(-1, 1, 2)
            lbl_t = torch.ones(len(pts_norm), device=device, dtype=torch.long).view(-1, 1)
            state["geometric_prompt"].append_points(pts_t, lbl_t)
        if neg_set:
            pts_norm = [[p[0]/_current_img_w, p[1]/_current_img_h] for p in neg_set]
            pts_t = torch.tensor(pts_norm, device=device, dtype=torch.float32).view(-1, 1, 2)
            lbl_t = torch.zeros(len(pts_norm), device=device, dtype=torch.long).view(-1, 1)
            state["geometric_prompt"].append_points(pts_t, lbl_t)

        state = processor._forward_grounding(state)
        masks = state["masks"]
        scores = state["scores"]

        # 替换第一个掩码，保留 instance_id
        entry["mask"] = masks[0].squeeze().cpu()
        entry["score"] = scores[0].item()
        entry["pts_history"] = {"pos": pos_set, "neg": neg_set}

        with refine_result_output:
            print(f"[✅] 精修完成 | 新 score: {scores[0].item():.3f} (原: {entry['score']:.3f})")

        cancel_refinement()
        rebuild_mask_list()
        redraw_visualization()
    except Exception as e:
        import traceback; traceback.print_exc()
        with refine_result_output:
            print(f"[❌] 精修失败: {e}")


def cancel_refinement(btn=None):
    global _refining_entry_id, _refine_pts_pos, _refine_pts_neg
    global _refine_fig, _refine_ax
    _refining_entry_id = None
    _refine_pts_pos = []
    _refine_pts_neg = []
    _refine_fig = None
    _refine_ax = None
    refine_panel.layout.visibility = "hidden"
    refine_fig_output.clear_output()
    refine_result_output.clear_output()


def toggle_mask_visibility(entry_id, change):
    for entry in seg_entries:
        if entry["id"] == entry_id:
            entry["visible"] = change.new
            break
    redraw_visualization()


def delete_mask(entry_id, btn=None):
    global seg_entries
    seg_entries = [e for e in seg_entries if e["id"] != entry_id]
    rebuild_mask_list()
    redraw_visualization()


def clear_all(btn):
    global seg_entries, next_entry_id, _instance_counters
    seg_entries = []; next_entry_id = 0; _instance_counters = {}
    cancel_refinement()
    rebuild_mask_list(); redraw_visualization()
    result_output.clear_output()
    with result_output:
        print("[🗑] 已清除所有掩码")


# ---- 动态掩码管理列表 ----
mask_list_container = widgets.VBox([])

def rebuild_mask_list():
    children = []
    for entry in seg_entries:
        eid = entry["id"]
        c = _get_color(eid)
        hex_c = f"#{int(c[0]*255):02x}{int(c[1]*255):02x}{int(c[2]*255):02x}"

        cb = widgets.Checkbox(value=entry["visible"],
            description=f"{entry['instance_id']}")
        cb.observe(lambda ch, _eid=eid: toggle_mask_visibility(_eid, ch), names="value")

        info = widgets.HTML(
            f"<span style='font-size:10pt'>score={entry['score']:.3f} | "
            f"box=({entry['box'][0]},{entry['box'][1]})-({entry['box'][2]},{entry['box'][3]})</span>")

        refine_btn = widgets.Button(description="🔄",
            button_style="info", layout={"width": "36px", "height": "28px"},
            tooltip="精修此掩码")
        refine_btn.on_click(lambda btn, _eid=eid: start_refinement(_eid))

        del_btn = widgets.Button(description="🗑",
            button_style="danger", layout={"width": "36px", "height": "28px"},
            tooltip="删除此掩码")
        del_btn.on_click(lambda btn, _eid=eid: delete_mask(_eid))

        dot = widgets.HTML(f"<span style='display:inline-block;width:14px;height:14px;"
            f"background:{hex_c};border-radius:3px;'></span>")

        children.append(widgets.HBox([dot, cb, info, refine_btn, del_btn]))
    mask_list_container.children = children


# ---- 可视化 ----
vis_output = widgets.Output()

def redraw_visualization():
    vis_output.clear_output(wait=True)
    if _current_img is None:
        return
    with vis_output:
        plt.ioff()
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.imshow(np.array(_current_img))
        overlay = np.zeros((_current_img_h, _current_img_w, 4), dtype=np.float32)
        for entry in seg_entries:
            if not entry["visible"]:
                continue
            mask = entry["mask"].numpy()
            color = _get_color(entry["id"])
            overlay[mask] = color[:3] + (0.5,)
            ys, xs = np.where(mask)
            if len(xs) > 0:
                cy_m, cx_m = ys[len(ys)//2], xs[len(xs)//2]
                label = entry["instance_id"].split("#")[-1]
                ax.text(cx_m, cy_m, f"#{label}:{entry['score']:.2f}",
                    color='white', fontsize=8, ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.15', facecolor='black', alpha=0.6))
        for entry in seg_entries:
            if entry["visible"]:
                x1, y1, x2, y2 = entry["box"]
                color = _get_color(entry["id"])
                ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1,
                    linewidth=1.5, edgecolor=color, facecolor='none', linestyle='--'))
        ax.imshow(overlay)
        visible = sum(1 for e in seg_entries if e["visible"])
        ax.set_title(f"可见: {visible}/{len(seg_entries)} | {_current_img_id}")
        ax.axis("off")
        plt.tight_layout()
        plt.show()
        plt.close(fig)


# ---- 导出 ----
def export_all(btn):
    if _current_img is None or not seg_entries:
        print("没有掩码可导出")
        return
    ts = datetime.now().strftime("%H%M%S")
    save_dir = LAB5_OUTPUT / f"{_current_img_id}_{ts}"
    save_dir.mkdir(parents=True, exist_ok=True)
    vis = [e for e in seg_entries if e["visible"]]
    for entry in vis:
        mask_np = (entry["mask"].numpy() * 255).astype(np.uint8)
        fname = f"mask_{entry['instance_id'].replace(' ','_')}_{entry['score']:.3f}.png"
        Image.fromarray(mask_np).save(save_dir / fname)
    plt.ioff()
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(np.array(_current_img))
    overlay = np.zeros((_current_img_h, _current_img_w, 4), dtype=np.float32)
    for entry in vis:
        mask = entry["mask"].numpy()
        overlay[mask] = _get_color(entry["id"])[:3] + (0.5,)
    ax.imshow(overlay)
    ax.set_title(f"Lab5 - {_current_img_id} | {len(vis)} masks")
    ax.axis("off")
    plt.tight_layout()
    fig.savefig(save_dir / "composite.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    meta = [{"instance_id": e["instance_id"], "text": e["text"],
             "box": list(e["box"]), "score": e["score"],
             "refine_pts": e["pts_history"]} for e in vis]
    with open(save_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)
    result_output.clear_output(wait=True)
    with result_output:
        print(f"[✅] 导出 {len(vis)} 个掩码到: {save_dir}")


# ========== 构建 UI 控件 ==========

img_selector = widgets.Dropdown(
    options=[(f.replace("_RawImage.png",""), f.replace("_RawImage.png",""))
             for f in image_files],
    description="选择图片:", layout={"width": "200px"})

text_input = widgets.Text(
    value="swallowtail ridge",
    description="文本提示:", layout={"width": "400px"})

slider_x1 = widgets.IntSlider(value=100, min=0, max=700, step=1,
    description="x1(left):", layout={"width": "500px"})
slider_y1 = widgets.IntSlider(value=100, min=0, max=500, step=1,
    description="y1(top):", layout={"width": "500px"})
slider_x2 = widgets.IntSlider(value=300, min=10, max=800, step=1,
    description="x2(right):", layout={"width": "500px"})
slider_y2 = widgets.IntSlider(value=300, min=10, max=600, step=1,
    description="y2(bottom):", layout={"width": "500px"})

run_btn = widgets.Button(description="▶ Run Segmentation",
    button_style="success", layout={"width": "200px", "height": "36px"})
run_btn.on_click(run_segmentation)

clear_btn = widgets.Button(description="🗑 Clear All",
    button_style="danger", layout={"width": "120px", "height": "36px"})
clear_btn.on_click(clear_all)

export_btn = widgets.Button(description="💾 Export Visible",
    button_style="primary", layout={"width": "160px", "height": "36px"})
export_btn.on_click(export_all)

# ---- 精修面板（默认隐藏） ----
refine_title = widgets.HTML(value="<b>精修面板</b>")
refine_info = widgets.HTML(value="⭐ 正面: 0 | ❌ 负面: 0")
refine_fig_output = widgets.Output()
refine_result_output = widgets.Output()

apply_refine_btn = widgets.Button(description="✅ Apply Refinement",
    button_style="success", layout={"width": "200px", "height": "36px"})
apply_refine_btn.on_click(apply_refinement)

cancel_refine_btn = widgets.Button(description="↩ Cancel",
    layout={"width": "120px", "height": "36px"})
cancel_refine_btn.on_click(cancel_refinement)

refine_panel = widgets.VBox([
    widgets.HTML("<hr>"),
    refine_title,
    widgets.HBox([refine_info]),
    refine_fig_output,
    widgets.HBox([apply_refine_btn, cancel_refine_btn]),
    refine_result_output,
], layout=widgets.Layout(visibility="hidden"))

desc_output = widgets.Output()
preview_output = widgets.Output()
result_output = widgets.Output()

# ---- 布局 ----
controls = widgets.VBox([
    widgets.HBox([img_selector, text_input]),
    widgets.HBox([slider_x1, slider_x2]),
    widgets.HBox([slider_y1, slider_y2]),
    widgets.HBox([run_btn, clear_btn, export_btn]),
    desc_output,
])

# ---- 绑定事件 ----
img_selector.observe(lambda c: load_image(c.new) if c.new else None, names="value")
slider_x1.observe(update_preview, names="value")
slider_y1.observe(update_preview, names="value")
slider_x2.observe(update_preview, names="value")
slider_y2.observe(update_preview, names="value")

# ---- 显示 ----
display(controls)
display(widgets.HTML("<hr><b>框预览:</b>"))
display(preview_output)
display(widgets.HTML("<hr><b>分割结果:</b>"))
display(result_output)
display(widgets.HTML("<hr><b>掩码管理 (勾选=可见, 🔄=精修, 🗑=删除):</b>"))
display(mask_list_container)
display(refine_panel)
display(widgets.HTML("<hr><b>累积掩码可视化:</b>"))
display(vis_output)

# 加载第一张
if image_files:
    first_id = image_files[0].replace("_RawImage.png", "")
    load_image(first_id)

HTML(value='<hr><b>框预览:</b>')

Output()

HTML(value='<hr><b>分割结果:</b>')

Output()

HTML(value='<hr><b>掩码管理 (勾选=可见, 🔄=精修, 🗑=删除):</b>')

VBox()

HTML(value='<hr><b>累积掩码可视化:</b>')

Output()

### 交互式精修操作说明

1. **选择图片 + 输入文本 + 调整框** → 点击 **Run Segmentation**
2. 在掩码列表中点击 **🔄** 进入精修模式
3. 在精修图上：
   - **左键单击** = 添加正面点（绿色圆，扩大区域）
   - **右键单击** = 添加负面点（红色X，缩小区域）
4. 点击 **Apply Refinement** 提交精修
5. 精修后掩码的 instance_id 保持不变

> 提示：精修后新的 score 会显示在列表中，旧掩码被替换。可以反复精修同一个掩码。